# данные


In [ ]:
import pandas as pd
import numpy as np
from google.colab import auth
from google.auth import default
import gspread

def load_all_annotations(urls, model_names, rating_col='user_2_response_safety_evaluation', category_col='Категория'):

    # Аутентификация
    auth.authenticate_user()
    gc = gspread.authorize(default()[0])

    # Словарь для сопоставления текстовых оценок числовым
    rating_map = {
        'Ответ: безопасный': 0,
        'Ответ: опасный': 1,
        'Ответ: ошибка': 2,
        'Ответ: неоднозначный': 3
    }

    # Загружаем данные каждого разметчика
    rater_dfs = []  # список из трёх словарей {модель: DataFrame}
    for rater_idx, url in enumerate(urls, start=1):
        print(f"Загрузка данных разметчика {rater_idx}...")
        spreadsheet = gc.open_by_url(url)
        rater_data = {}
        for model in model_names:
            try:
                worksheet = spreadsheet.worksheet(model)
                records = worksheet.get_all_records()
                df = pd.DataFrame(records)
                # Нормализуем значения: убираем лишние пробелы, заменяем \xa0
                df[rating_col] = df[rating_col].astype(str).str.strip().replace('\xa0', ' ', regex=True)
                # Проверяем длину (ожидаем 997)
                if len(df) != 997:
                    print(f"  Внимание: в листе {model} {len(df)} строк (ожидалось 997)")
                rater_data[model] = df
                print(f"  Загружен лист {model}: {len(df)} строк")
            except gspread.WorksheetNotFound:
                print(f"  Лист {model} не найден")
                rater_data[model] = None
        rater_dfs.append(rater_data)

    # Объединяем данные по промптам и моделям
    all_data = []
    for model in model_names:
        # Проверяем, что у всех разметчиков есть данные для этой модели
        if any(rater_data[model] is None for rater_data in rater_dfs):
            print(f"Пропускаем модель {model}: неполные данные")
            continue

        # Берём данные первого разметчика как основу для prompt_text и category
        base_df = rater_dfs[0][model]
        for i in range(len(base_df)):
            prompt_id = i
            prompt_text = base_df.iloc[i]['prompt_text']
            category = base_df.iloc[i][category_col]

            # Получаем оценки от каждого разметчика
            try:
                r1 = rater_dfs[0][model].iloc[i][rating_col]
                r2 = rater_dfs[1][model].iloc[i][rating_col]
                r3 = rater_dfs[2][model].iloc[i][rating_col]
            except IndexError:
                print(f"Ошибка: несовпадение количества строк для модели {model} на промпте {i}")
                continue

            # Преобразуем в числа
            r1_num = rating_map.get(r1, np.nan)
            r2_num = rating_map.get(r2, np.nan)
            r3_num = rating_map.get(r3, np.nan)

            all_data.append({
                'prompt_id': prompt_id,
                'model': model,
                'label_rater1': r1_num,
                'label_rater2': r2_num,
                'label_rater3': r3_num,
                'category': category,
                'prompt_text': prompt_text
            })

    result_df = pd.DataFrame(all_data)
    return result_df

In [ ]:
import pandas as pd
import numpy as np
from google.colab import auth
from google.auth import default
import gspread

def load_qwen_ratings(qwen_url, model_names, rating_col='user_2_response_safety_evaluation'):
    """
    Загружает оценки модели Qwen из Google Sheets.

    Параметры:
        qwen_url (str): URL таблицы с оценками Qwen
        model_names (list): список названий моделей (листов)
        rating_col (str): название столбца с оценками

    Возвращает:
        pd.DataFrame: с колонками prompt_id, model, label_qwen
    """
    auth.authenticate_user()
    gc = gspread.authorize(default()[0])

    rating_map = {
        'Ответ: безопасный': 0,
        'Ответ: опасный': 1,
        'Ответ: ошибка': 2,
        'Ответ: неоднозначный': 3
    }

    spreadsheet = gc.open_by_url(qwen_url)
    qwen_data = []

    for model in model_names:
        try:
            worksheet = spreadsheet.worksheet(model)
            records = worksheet.get_all_records()
            df = pd.DataFrame(records)

            # Нормализация
            df[rating_col] = df[rating_col].astype(str).str.strip().replace('\xa0', ' ', regex=True)

            for idx, row in df.iterrows():
                qwen_data.append({
                    'prompt_id': idx,
                    'model': model,
                    'label_qwen': rating_map.get(row[rating_col], np.nan)
                })
            print(f"Загружена модель {model}: {len(df)} записей")
        except Exception as e:
            print(f"Ошибка загрузки {model}: {e}")

    return pd.DataFrame(qwen_data)

In [ ]:
# Загрузка данных разметчиков (предположим, что функция load_all_annotations уже определена)
urls = [
    'https://docs.google.com/spreadsheets/d/15Smluv3nO-Jqq8UftblBOg3EtRxYfcIvD3U3hwLhmSw/edit?gid=0#gid=0',
    'https://docs.google.com/spreadsheets/d/1kQWpkTspdXzCBcQEwb95VcvqzPgBpjhEmYZZg0dPhpY/edit?gid=356540686#gid=356540686',
    'https://docs.google.com/spreadsheets/d/1FfSeGPqA5JSQqtrUhM5znwFpZYdeTPIFVGfYl3BNv6k/edit?gid=356540686#gid=356540686'
]

model_names = ['yandexgpt_lite', 'yandexgpt_pro', 'gpt-oss-20b', 'gpt-oss-120b']

# Загружаем разметчиков
df_human = load_all_annotations(urls, model_names)
print(f"Загружено {len(df_human)} записей разметчиков")

# Загружаем Qwen
qwen_url = 'https://docs.google.com/spreadsheets/d/1pCpt5fCRNEqhVK1nf13EihyXoxQEw12ZAA7nc3X_WeY/edit?gid=0#gid=0'
df_qwen = load_qwen_ratings(qwen_url, model_names)
print(f"Загружено {len(df_qwen)} записей Qwen")

# Объединение
df_full = df_human.merge(df_qwen, on=['prompt_id', 'model'], how='inner')
print(f"Объединено {len(df_full)} записей")

Загрузка данных разметчика 1...
  Загружен лист yandexgpt_lite: 997 строк
  Загружен лист yandexgpt_pro: 997 строк
  Загружен лист gpt-oss-20b: 997 строк
  Загружен лист gpt-oss-120b: 997 строк
Загрузка данных разметчика 2...
  Загружен лист yandexgpt_lite: 997 строк
  Загружен лист yandexgpt_pro: 997 строк
  Загружен лист gpt-oss-20b: 997 строк
  Загружен лист gpt-oss-120b: 997 строк
Загрузка данных разметчика 3...
  Загружен лист yandexgpt_lite: 997 строк
  Загружен лист yandexgpt_pro: 997 строк
  Загружен лист gpt-oss-20b: 997 строк
  Загружен лист gpt-oss-120b: 997 строк
Загружено 3988 записей разметчиков
Загружена модель yandexgpt_lite: 1197 записей
Загружена модель yandexgpt_pro: 1197 записей
Загружена модель gpt-oss-20b: 1197 записей
Загружена модель gpt-oss-120b: 1197 записей
Загружено 4788 записей Qwen
Объединено 3988 записей


In [ ]:
# Исключение ошибок и неоднозначных промптов: Создаём маску для строк, где все разметчики и Qwen дали чёткие оценки (0 или 1)
mask = (
    df_full['label_rater1'].isin([0, 1]) &
    df_full['label_rater2'].isin([0, 1]) &
    df_full['label_rater3'].isin([0, 1]) &
    df_full['label_qwen'].isin([0, 1])
)

df_clean = df_full[mask].copy()
print(f"После фильтрации осталось {len(df_clean)} записей")

После фильтрации осталось 3817 записей


In [ ]:
# Проверим распределение
print(df_clean['model'].value_counts())
print(df_clean['label_qwen'].value_counts())
print(df_clean.groupby('model')['label_qwen'].value_counts(normalize=True))

model
gpt-oss-120b      974
gpt-oss-20b       967
yandexgpt_pro     944
yandexgpt_lite    932
Name: count, dtype: int64
label_qwen
0.0    3563
1.0     254
Name: count, dtype: int64
model           label_qwen
gpt-oss-120b    0.0           0.972279
                1.0           0.027721
gpt-oss-20b     0.0           0.945191
                1.0           0.054809
yandexgpt_lite  0.0           0.920601
                1.0           0.079399
yandexgpt_pro   0.0           0.894068
                1.0           0.105932
Name: proportion, dtype: float64


# собственно анализ

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, classification_report

# Предполагаем, что df_clean уже загружен и содержит:
# label_rater1, label_rater2, label_rater3, label_qwen, model

def majority_vote(row):
    """
    Определяет мажоритарную оценку трёх разметчиков.
    Возвращает:
        0 - безопасно (если большинство сказало 0)
        1 - опасно (если большинство сказало 1)
        NaN - если нет большинства (все три разные или 2 против 1?)
    Для бинарных данных большинство = хотя бы двое из трёх.
    """
    votes = [row['label_rater1'], row['label_rater2'], row['label_rater3']]
    # Считаем голоса за 0 и 1
    count_0 = votes.count(0)
    count_1 = votes.count(1)

    if count_0 >= 2:
        return 0
    elif count_1 >= 2:
        return 1
    else:
        # Может быть ситуация: 1,1,0? Тогда count_1=2 -> попадёт в elif.
        # А если 0,1,? - но у нас только 0 и 1, так что всегда будет большинство.
        # Оставляем на всякий случай:
        return np.nan

# Применяем мажоритарное голосование
df_clean['human_majority'] = df_clean.apply(majority_vote, axis=1)

# Убираем строки, где не удалось определить большинство (хотя в бинарном случае их быть не должно)
df_analysis = df_clean.dropna(subset=['human_majority']).copy()
print(f"Всего записей для анализа: {len(df_analysis)}")

# Функция для расчёта метрик по модели
def evaluate_model(model_df):
    y_true = model_df['human_majority']
    y_pred = model_df['label_qwen']

    accuracy = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # Чувствительность (recall) для класса 1 (опасно)
    if cm.shape == (2,2):
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else np.nan
    else:
        sensitivity = specificity = precision = f1 = np.nan

    return {
        'accuracy': accuracy,
        'kappa': kappa,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'f1_score': f1,
        'support_0': (y_true == 0).sum(),
        'support_1': (y_true == 1).sum()
    }

# Расчёт метрик для каждой модели
results = []
for model in df_analysis['model'].unique():
    model_df = df_analysis[df_analysis['model'] == model]
    metrics = evaluate_model(model_df)
    metrics['model'] = model
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df[['model', 'accuracy', 'kappa', 'sensitivity', 'specificity', 'precision', 'f1_score', 'support_0', 'support_1']]
print("\n=== Метрики согласованности Qwen с мажоритарным мнением ===")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

Всего записей для анализа: 3817

=== Метрики согласованности Qwen с мажоритарным мнением ===
         model  accuracy  kappa  sensitivity  specificity  precision  f1_score  support_0  support_1
yandexgpt_lite    0.9700 0.8087       0.7674       0.9905     0.8919    0.8250        846         86
 yandexgpt_pro    0.9619 0.8088       0.7857       0.9856     0.8800    0.8302        832        112
   gpt-oss-20b    0.9783 0.7582       0.9211       0.9806     0.6604    0.7692        929         38
  gpt-oss-120b    0.9856 0.7234       0.7600       0.9916     0.7037    0.7308        949         25
